# 05 域内测试与跨场景评价

**目标：** 对事先锁定的实验模型，统一评价 PlantVillage 域内测试与 PlantDoc 外部测试，比较泛化差值。

**前置条件：** E1—E4 中计划进行的训练和模型选择已结束；固定测试清单已完成；所有待评价模型已训练。

**输入：** `experiments.yaml` 中的 `evaluation_runs` 和 `evaluation_locked`、候选实验 `best.pt`、两份固定测试清单。

**输出：** 每个 run 的 `metrics.json`、`predictions.csv`，以及 `outputs/tables/cross_domain_comparison.csv`。

默认关闭评价。先填写完整的候选实验目录名列表，再将 `evaluation_locked` 设为 `true`；外部测试结果用于最终报告，不用于回头选择增强或调参。

In [ ]:
from pathlib import Path
from copy import deepcopy
import csv
import html
import json
import sys


def find_notebook_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        for root in (candidate, candidate / "plant-disease-project"):
            if (root / "configs" / "base.yaml").is_file():
                return root
    raise FileNotFoundError("请从项目根目录或其 notebooks 目录启动 Jupyter。")


PROJECT_ROOT = find_notebook_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_config

BASE = load_config(PROJECT_ROOT / "configs" / "base.yaml")
EXPERIMENTS = load_config(PROJECT_ROOT / "configs" / "experiments.yaml")
LABELS = load_config(PROJECT_ROOT / "configs" / "labels.yaml")


def read_csv(path):
    path = Path(path)
    if not path.exists():
        return []
    with path.open(encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def show_table(records, columns=None, limit=20):
    records = list(records)
    if not records:
        print("暂无记录。")
        return
    columns = columns or list(records[0])
    rows = records[:limit]
    header = "".join(f"<th>{html.escape(str(c))}</th>" for c in columns)
    body = "".join("<tr>" + "".join(
        f"<td>{html.escape(str(row.get(c, '')))}</td>" for c in columns
    ) + "</tr>" for row in rows)
    try:
        from IPython.display import HTML, display
        display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))
    except ImportError:
        print(json.dumps(rows, ensure_ascii=False, indent=2))
    if len(records) > limit:
        print(f"仅展示前 {limit} 行，共 {len(records)} 行。")


print("项目目录：", PROJECT_ROOT)
print("已配置类别数：", len(LABELS.get("classes", [])))

## 1. 审核并锁定评价计划

`evaluation_runs` 示例形式为 `e1_resnet18_seed42`，必须与 `outputs/runs/` 中真实目录一致。既可以纳入基线和最终优化模型，也可以纳入预先约定的对照组；请在看到外测结果前决定。

In [ ]:
RUN_EVALUATION = False
locked = EXPERIMENTS.get("evaluation_locked") is True
run_ids = EXPERIMENTS.get("evaluation_runs", [])
if not isinstance(run_ids, list) or not all(isinstance(name, str) for name in run_ids):
    raise ValueError("evaluation_runs 必须是实验目录名组成的列表。")

evaluation_plan = []
for run_id in run_ids:
    run_dir = PROJECT_ROOT / "outputs" / "runs" / run_id
    evaluation_plan.append({"run_id": run_id, "checkpoint_exists": (run_dir / "best.pt").is_file(),
                            "config_exists": (run_dir / "config.yaml").is_file()})
show_table(evaluation_plan)
ready = locked and bool(run_ids) and all(row["checkpoint_exists"] and row["config_exists"]
                                         for row in evaluation_plan)
print("评价候选是否锁定：", locked)
if not run_ids:
    print("请先将所有待评价实验写入 configs/experiments.yaml 的 evaluation_runs。")
elif not locked:
    print("尚未锁定候选列表；确认后设置 evaluation_locked: true。")
elif not ready:
    print("部分实验权重或配置缺失，请先完成相应训练。")
else:
    print("候选已锁定且模型文件齐全；确认后可开启 RUN_EVALUATION。")

## 2. 运行统一评价

本单元先检查所有候选与测试清单，再评价两个测试集。评价使用确定性预处理；类别顺序必须与模型保存的标签一致。写出的逐图预测便于后续错误分析。

In [ ]:
evaluation_results = []
if not RUN_EVALUATION:
    print("尚未运行评价；目前不会读取 PlantDoc 外部测试样本。")
elif not ready:
    print("评价前置条件未满足；请先锁定完整候选并补齐模型。")
else:
    from src.evaluate import evaluate_checkpoint
    test_manifests = {
        "plantvillage_test": BASE["data"]["test_manifest"],
        "plantdoc_external_test": BASE["data"]["external_manifest"],
    }
    missing = [path for path in test_manifests.values() if not (PROJECT_ROOT / path).is_file()]
    if missing:
        raise FileNotFoundError(f"固定测试清单缺失：{missing}。请先完成 Notebook 01。")
    for run_id in run_ids:
        run_dir = PROJECT_ROOT / "outputs" / "runs" / run_id
        for dataset_name, manifest in test_manifests.items():
            metrics = evaluate_checkpoint(PROJECT_ROOT, run_dir, manifest, dataset_name=dataset_name)
            evaluation_results.append({"run_id": run_id, **metrics})
            print(run_id, dataset_name, "Accuracy =", round(metrics["accuracy"], 4),
                  "Macro-F1 =", round(metrics["macro_f1"], 4))

## 3. 汇总已锁定模型的真实结果

泛化差值定义为“PlantVillage 指标减 PlantDoc 指标”；正值越大，跨场景性能下降越明显。不同测试集类别分布可能不同，应同时报告 Macro-F1、各类 Recall 和样本数，避免仅比较 Accuracy。

下表中的推理耗时为当前设备上的批次摊销前向耗时，不能当作单张端到端延迟；需要部署效率比较时另行使用统一的单张推理基准。

In [ ]:
comparison = []
if ready:
    for run_id in run_ids:
        metrics_path = PROJECT_ROOT / "outputs" / "runs" / run_id / "metrics.json"
        if not metrics_path.exists():
            continue
        metrics = load_config(metrics_path)
        pv = metrics.get("plantvillage_test")
        pd = metrics.get("plantdoc_external_test")
        if pv and pd:
            comparison.append({
                "run_id": run_id,
                "pv_samples": pv["sample_count"], "pd_samples": pd["sample_count"],
                "pv_accuracy": pv["accuracy"], "pd_accuracy": pd["accuracy"],
                "accuracy_gap": pv["accuracy"] - pd["accuracy"],
                "pv_macro_f1": pv["macro_f1"], "pd_macro_f1": pd["macro_f1"],
                "macro_f1_gap": pv["macro_f1"] - pd["macro_f1"],
                "checkpoint_bytes": pv.get("checkpoint_bytes"),
                "pv_batch_amortized_inference_ms": pv.get("batch_amortized_inference_ms"),
                "pd_batch_amortized_inference_ms": pd.get("batch_amortized_inference_ms"),
                "device": pv.get("device"),
            })
show_table(comparison)
if comparison:
    output_path = PROJECT_ROOT / "outputs" / "tables" / "cross_domain_comparison.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(comparison[0]))
        writer.writeheader()
        writer.writerows(comparison)
    print("汇总表已保存：", output_path)
else:
    print("尚无完整且已锁定的域内/外部测试结果，不生成汇总表。")

## 结论记录

- 评价前锁定的候选列表与日期：待填写。
- Accuracy、Macro-F1、逐类 Recall 及随机种子波动：待实际评价后填写。
- 域内与外部测试的差值及原因分析：待填写。
- 样本量、类别分布、标注质量与测试集使用的局限：待填写。
- 外测后发现的问题只记录为后续研究，不继续用本次外测调参。